Ultralytics wraps everything model loading, inference, result parsing
Loads YOLOv8n. First time you run this, it downloads the weights (~6MB) from NVIDIA's servers and caches them locally. Every run after that loads from cache. This is equivalent to loading a .plan file on DRIVE Orin at boot time — happens once, stays in memory.


In [10]:
from ultralytics import YOLO

# Load YOLOv8n — downloads ~6MB on first run, cached after
model = YOLO('yolov8n.pt')

# Run inference on sample street scene — device='mps' forces M4 GPU
results = model.predict('https://ultralytics.com/images/bus.jpg', device='mps')

# Print every detected object
for box in results[0].boxes:
    cls   = int(box.cls[0])
    conf  = float(box.conf[0])
    label = model.names[cls]
    print(f"{label}: {conf:.2f}")


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /Users/manojkumar/RoadMap/ml-automotive-foundations/edge-inference/bus.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 10.9ms
Speed: 1.7ms preprocess, 10.9ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 480)
bus: 0.87
person: 0.87
person: 0.85
person: 0.83
person: 0.26
stop sign: 0.26


In [11]:
# Save annotated image with bounding boxes drawn
results[0].save(filename='p1_4_result.jpg')
print("Saved: p1_4_result.jpg")

Saved: p1_4_result.jpg


In [12]:
import time

# Warmup — discard first 10 runs
for _ in range(10):
    model.predict('https://ultralytics.com/images/bus.jpg', 
                  device='mps', verbose=False)

# Benchmark — 50 runs
times = []
for _ in range(50):
    start = time.perf_counter()
    model.predict('https://ultralytics.com/images/bus.jpg', 
                  device='mps', verbose=False)
    end = time.perf_counter()
    times.append((end - start) * 1000)  # convert to ms

mean_ms = sum(times) / len(times)
std_ms  = (sum((t - mean_ms)**2 for t in times) / len(times)) ** 0.5

print(f"YOLOv8n on M4 (MPS)")
print(f"Mean latency : {mean_ms:.2f} ms")
print(f"Std deviation: {std_ms:.2f} ms")

Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus

In [13]:
# Export YOLOv8n to ONNX
path = model.export(format='onnx')
print(f"ONNX model saved to: {path}")

Ultralytics 8.4.54 🚀 Python-3.12.1 torch-2.10.0 CPU (Apple M4)

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.71'] not found, attempting AutoUpdate...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxslim]


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

requirements: AutoUpdate success ✅ 1.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.6s, saved as 'yolov8n.onnx' (12.3 MB)

Export complete (2.8s)
Results saved to /Users/manojkumar/RoadMap/ml-automotive-foundations/edge-inference/yolov8n.onnx
Predict:         yolo predict task=detect model=yolov8n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolov8n.onnx imgsz=640 data=coc

In [1]:
import os

size_mb = os.path.getsize('yolov8n.onnx') / (1024 * 1024)
print(f"YOLOv8n ONNX size: {size_mb:.1f} MB")

YOLOv8n ONNX size: 12.3 MB
